# YogaPose StyleGAN2-ADA Colab (A100-ready)

This notebook trains a **StyleGAN2-ADA** model on a **subset of 5–10 yoga pose classes** and then **generates + displays** synthetic samples directly inside the notebook.

## Recommended runtime
- **GPU:** NVIDIA A100 preferred
- TPU is **not recommended** for this workflow

## What this notebook does
1. Mounts Google Drive
2. Downloads / reuses the `arrowe/yoga-poses-dataset-107` zip on Drive
3. Extracts dataset and selects 5–10 yoga pose classes
4. Builds a StyleGAN2-ADA training dataset ZIP
5. Clones StyleGAN2-ADA
6. Launches training with Drive-backed outputs
7. Generates images after training
8. **Displays generated images inline** + prints output paths

> Note: a short run under ~30 minutes produces a **prototype** model, not a fully converged generator.

---
## Cell 1 — Mount Drive and configure paths

All outputs (subset dataset, training runs, generated samples) are saved to Google Drive so you can resume even if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Root folder for this notebook's outputs
DRIVE_ROOT = '/content/drive/MyDrive/YogaPoseStyleGAN'

# Dataset zip downloaded earlier
DATA_ZIP = '/content/drive/MyDrive/computer-vision/yoga-poses-dataset-107.zip'

# Where we extract the Kaggle zip
EXTRACT_DIR = os.path.join(DRIVE_ROOT, 'dataset')

# Where we write a resized, class-filtered subset
SUBSET_DIR = os.path.join(DRIVE_ROOT, 'subset_dataset')

# Where StyleGAN2-ADA code will be cloned
STYLEGAN_REPO = '/content/stylegan2-ada-pytorch'

# Training dataset zip consumed by StyleGAN2-ADA
TRAIN_ZIP = os.path.join(DRIVE_ROOT, 'yoga_subset_stylegan.zip')

# Training outputs / checkpoints
RUNS_DIR = os.path.join(DRIVE_ROOT, 'training-runs')

# Where we save images generated by generate.py
GENERATED_DIR = os.path.join(DRIVE_ROOT, 'generated_samples')

for d in [DRIVE_ROOT, EXTRACT_DIR, SUBSET_DIR, RUNS_DIR, GENERATED_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted ✓')
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('DATA_ZIP     =', DATA_ZIP)
print('EXTRACT_DIR  =', EXTRACT_DIR)
print('SUBSET_DIR   =', SUBSET_DIR)
print('TRAIN_ZIP    =', TRAIN_ZIP)
print('RUNS_DIR     =', RUNS_DIR)
print('GENERATED_DIR=', GENERATED_DIR)

---
## Cell 2 — Install dependencies

We only need a few utilities for preprocessing + visualization. StyleGAN2-ADA will install its own Python requirements from the repo.

In [ ]:
!pip install -q pillow tqdm matplotlib

---
## Cell 3 — Download dataset zip if needed

This uses the Kaggle endpoint you provided. If Kaggle requires auth/cookies in your environment, you may need to download manually from Kaggle and place the zip at `DATA_ZIP`.

In [ ]:
if not os.path.exists(DATA_ZIP):
    print('Downloading dataset zip...')
    !curl -L https://www.kaggle.com/api/v1/datasets/download/arrowe/yoga-poses-dataset-107 -o "{DATA_ZIP}"
else:
    print('Dataset zip already exists ✓')

print('Zip path:', DATA_ZIP)

---
## Cell 4 — Extract dataset and discover train directory

In [ ]:
import glob, zipfile

train_dirs = glob.glob(os.path.join(EXTRACT_DIR, '**/train'), recursive=True)
if not train_dirs:
    print('Extracting dataset...')
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print('Dataset already extracted ✓')

train_dirs = glob.glob(os.path.join(EXTRACT_DIR, '**/train'), recursive=True)
DATASET_DIR = train_dirs[0] if train_dirs else EXTRACT_DIR
print('DATASET_DIR =', DATASET_DIR)

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print('Total classes found:', len(classes))
print('Example classes:', classes[:20])

---
## Cell 5 — Choose 5–10 yoga poses and image size

- `IMG_SIZE=256` is recommended on A100.
- If you run out of time/memory, switch to `IMG_SIZE=128`.

Important: the class names must exactly match the folder names printed in Cell 4.

In [ ]:
# ⚙️ Edit these to match your dataset folder names
SELECTED_CLASSES = [
    # Update to match DATASET_DIR folder names
    'adho mukha svanasana',
    'vrksasana',
    'trikonasana',
    'virabhadrasana ii',
    'balasana',
    'bhujangasana',
    'dhanurasana',
    'bakasana'
]

IMG_SIZE = 256
MAX_IMAGES_PER_CLASS = 300

print('Selected classes:', SELECTED_CLASSES)
print('IMG_SIZE:', IMG_SIZE)
print('MAX_IMAGES_PER_CLASS:', MAX_IMAGES_PER_CLASS)

---
## Cell 6 — Build subset dataset (copy + resize to square)

StyleGAN training benefits from consistent input sizes. We resize while preserving aspect ratio, then pad to a square canvas.

In [ ]:
import shutil
from PIL import Image, ImageOps
from tqdm import tqdm

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# Clear previous subset so the run is reproducible
if os.path.exists(SUBSET_DIR):
    shutil.rmtree(SUBSET_DIR)
os.makedirs(SUBSET_DIR, exist_ok=True)

def resize_and_pad(img, size):
    img = ImageOps.exif_transpose(img).convert('RGB')
    img.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), (255, 255, 255))
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

summary = {}
for cls in SELECTED_CLASSES:
    src_dir = os.path.join(DATASET_DIR, cls)
    dst_dir = os.path.join(SUBSET_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        print(f'WARNING: class not found: {cls}')
        summary[cls] = 0
        continue

    files = [f for f in sorted(os.listdir(src_dir)) if os.path.splitext(f)[1].lower() in VALID_EXTS]
    files = files[:MAX_IMAGES_PER_CLASS]

    count = 0
    for fname in tqdm(files, desc=cls):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dst_dir, os.path.splitext(fname)[0] + '.png')
        try:
            img = Image.open(src)
            out = resize_and_pad(img, IMG_SIZE)
            out.save(dst, format='PNG')
            count += 1
        except Exception:
            pass

    summary[cls] = count

print('Subset build complete ✓')
print('Images per selected class:')
for k, v in summary.items():
    print(f'  {k:30s} {v}')

---
## Cell 7 — Create StyleGAN training ZIP

StyleGAN2-ADA can train from a folder or a zip. We create a zip for portability.

In [ ]:
import zipfile

if os.path.exists(TRAIN_ZIP):
    os.remove(TRAIN_ZIP)

with zipfile.ZipFile(TRAIN_ZIP, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, _, files in os.walk(SUBSET_DIR):
        for f in files:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, SUBSET_DIR)
            zf.write(full, rel)

print('Training zip created ✓')
print(TRAIN_ZIP)

---
## Cell 8 — Clone StyleGAN2-ADA

We use NVIDIA's official StyleGAN2-ADA PyTorch implementation.

In [ ]:
if not os.path.exists(STYLEGAN_REPO):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git {STYLEGAN_REPO}
else:
    print('StyleGAN2-ADA repo already cloned ✓')

%cd {STYLEGAN_REPO}

# Install StyleGAN requirements
!pip install -q -r requirements.txt

---
## Cell 9 — Check GPU + choose training settings

These are *prototype* settings aimed at producing samples quickly. For better results, increase `KIMG` (training duration).

In [ ]:
!nvidia-smi

# Fast prototype settings for ~<30 min A100 run (adjust as needed)
KIMG = 200         # thousands of images shown to discriminator
SNAP = 20          # snapshot interval
BATCH = 16         # batch size (try 32 on A100 if it fits)
AUG = 'ada'        # adaptive discriminator augmentation

print('Training settings:')
print('  KIMG =', KIMG)
print('  SNAP =', SNAP)
print('  BATCH=', BATCH)
print('  AUG  =', AUG)

---
## Cell 10 — Launch training

Outputs are saved under `RUNS_DIR` on Drive.

Tip: If you restart Colab, you can skip dataset prep and re-run from this cell to continue experimenting.

In [ ]:
!python train.py \
  --outdir={RUNS_DIR} \
  --data={TRAIN_ZIP} \
  --gpus=1 \
  --cfg=auto \
  --snap={SNAP} \
  --kimg={KIMG} \
  --batch={BATCH} \
  --aug={AUG}

---
## Cell 11 — Locate latest network snapshot

StyleGAN2-ADA writes snapshots like `network-snapshot-000200.pkl`. We'll find the most recent one.

In [ ]:
import glob

run_dirs = sorted(glob.glob(os.path.join(RUNS_DIR, '*')))
latest_run = run_dirs[-1] if run_dirs else None
print('Latest run:', latest_run)

snapshots = sorted(glob.glob(os.path.join(latest_run, 'network-snapshot-*.pkl'))) if latest_run else []
NETWORK_PKL = snapshots[-1] if snapshots else None
print('Latest snapshot:', NETWORK_PKL)

---
## Cell 12 — Generate sample images

We generate 25 images (seeds 0–24) and write them to Drive in `GENERATED_DIR`.

In [ ]:
if NETWORK_PKL is None:
    print('No snapshot found. Train first (Cell 10).')
else:
    # Clear existing generated samples (optional)
    for f in glob.glob(os.path.join(GENERATED_DIR, '*.png')):
        os.remove(f)

    !python generate.py --outdir={GENERATED_DIR} --trunc=0.8 --seeds=0-24 --network={NETWORK_PKL}
    print('Generated samples saved to:', GENERATED_DIR)

---
## Cell 13 — Display generated samples inline

This cell loads the `.png` images from Drive and displays them in a grid. This makes it easy to take screenshots / include results in your report.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_files = sorted(glob.glob(os.path.join(GENERATED_DIR, '*.png')))
print('Found', len(sample_files), 'generated images in', GENERATED_DIR)

if not sample_files:
    print('No generated samples found. Run Cell 12 first.')
else:
    # Show up to 25 images (5x5)
    n = min(25, len(sample_files))
    grid = int(n ** 0.5)
    grid = 5 if n >= 25 else max(1, grid)

    fig, axes = plt.subplots(grid, grid, figsize=(12, 12))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, f in zip(axes, sample_files[:grid*grid]):
        ax.imshow(mpimg.imread(f))
        ax.set_title(os.path.basename(f), fontsize=7)
        ax.axis('off')

    # Hide any unused axes
    for ax in axes[len(sample_files[:grid*grid]):]:
        ax.axis('off')
    
    plt.suptitle('Generated YogaPose Samples (StyleGAN2-ADA)', fontsize=14)
    plt.tight_layout()
    plt.show()

    print('Sample file list:')
    for f in sample_files[:10]:
        print(' ', f)
    if len(sample_files) > 10:
        print(' ...')